# Somo la 12 - Kupunguza Historia ya Gumzo kwa kutumia Daftari la Wakala

Kitabu hiki cha mazoezi kinaonyesha jinsi ya kusimamia muktadha katika mazungumzo marefu kwa kutumia Mfumo wa Wakala wa Microsoft. Kadiri mazungumzo yanavyokua, idadi ya tokeni huongezeka — hatimaye kupita dirisha la muktadha la modeli. Tunatatua hili kwa kutumia **mfano wa muhtasari wa muktadha** na **daftari la wakala** kwa kumbukumbu ya kudumu.

## Utajifunza:
1. **Kwa Nini Usimamizi wa Muktadha ni Muhimu**: Kuelewa viwango vya tokeni na dirisha za muktadha
2. **Wakala Wanaojua Muktadha**: Kuunda wakala wanaosimamia muktadha wao wa mazungumzo
3. **Mfano wa Muhtasari wa Muktadha**: Kutumia zana kufupisha historia ya mazungumzo
4. **Daftari la Wakala**: Kumbukumbu ya kudumu inayodumu baada ya kupunguzwa kwa muktadha

## Mahitaji ya Awali:
- Usanidi wa Azure OpenAI na vigezo vya mazingira vimewekwa
- Uelewa wa dhana za wakala za msingi kutoka masomo ya awali


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Kwa Nini Usimamizi wa Muktadha Ni Muhimu

Kila LLM ina **dirisha la muktadha** la mwisho — idadi kubwa ya tokeni inayoweza kusindika katika ombi moja. Kadri mazungumzo ya mzunguko mwingi yanavyosonga:

- **Idadi ya tokeni inaongezeka kwa mstari** kila ujumbe wa mtumiaji na jibu la msaidizi.
- **Tokeni za maagizo ndizo zinazochangia gharama kuu** kwa sababu historia yote hujazwa tena kila mzunguko.
- Hatimaye mazungumzo **huvuka dirisha la muktadha** na mfano au huchekesha au hutoa makosa.

### Mikakati ya Kusimamia Muktadha

| Mkakati | Inavyofanya Kazi | Upande wa Kukatiza |
|---|---|---|
| **Kukata** | Kuacha ujumbe wa zamani zaidi | Analeta upungufu wa muktadha wa mapema |
| **Muhtasari** | Kubana jumbe za zamani kuwa muhtasari | Baadhi ya maelezo hupotea, lakini hoja kuu zinahifadhiwa |
| **Mwandiko wa Kumbukumbu / Kumbukumbu ya Nje** | Kuhifadhi ukweli muhimu nje ya mazungumzo | Inahitaji simu za zana, lakini huishi kupungua kwa muktadha |

Katika daftari hili tunachanganya **muhtasari** na **zana ya mwandiko wa kumbukumbu** hivyo wakala anaweza kudumisha muendelezo hata wakati historia ya mazungumzo inabana.


## Kuunda Wakala Anayeielewa Muktadha


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Kuiga Mazungumzo Marefu

Hebu tugonge kupitia mazungumzo yenye mizunguko mingi kuona jinsi muktadha unavyojumuika. Mwakilishi anapaswa kuhifadhi maelezo muhimu (mapendeleo, bajeti, tarehe za kusafiri) katika mizunguko na kuonyesha muendelezo.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Angalia jinsi wakala anavyoendelea kuhifadhi muktadha kutoka kwa zamu za awali — anajua kuhusu Japan, sushi, mahekalu, upigaji picha, bajeti ya $3000, kusafiri peke yake, na muda wa Aprili. Katika mazungumzo mafupi hii hufanya kazi vizuri, lakini kadri mazungumzo yanavyoongezeka historia kamili huwa ghali tena kutumwa.

Tuendelee na mazungumzo kwa zamu zaidi ili kuona mkusanyiko wa muktadha:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Mfumo wa Muhtasari wa Muktadha

Kadhaa mazungumzo yanavyozidi, tunaweza kutumia **kifaa cha muhtasari** kukusanya muktadha uliokusanywa katika muundo mfupi. Wakala huuita kifaa hiki kurekodi mapendeleo muhimu ili hata kama ujumbe wa zamani utakaporomolewa, taarifa muhimu zibaki.

Mfumo huu ni msingi wa kupunguza historia kwa njia ya ufanisi zaidi:
1. Wakala hutambua ukweli muhimu kutoka mazungumzo
2. Huitumia kifaa cha muhtasari kuziweka
3. Ujumbe wa zamani unaweza kuondolewa kwa usalama kwa sababu muhtasari unahifadhi kinachohitajika

Hapa chini tunaelezea kifaa cha `summarize_preferences` ambacho wakala anaweza kuitumia kurekodi muhtasari mfupi wa kile alichojifunza.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Muhtasari

Katika somo hili ulijifunza jinsi ya kusimamia muktadha katika mazungumzo ya wakala wanaoendelea kwa muda mrefu kwa kutumia Microsoft Agent Framework:

### Dhahania Muhimu
- **Dirisha za muktadha ni za muda mfupi** — kila tokeni katika historia ya mazungumzo huchukua gharama na kuhesabiwa kuelekea kikomo.
- **Zana za muhtasari** huruhusu wakala kubana muktadha uliokusanywa kuwa muhtasari mfupi, kupunguza matumizi ya tokeni huku ikihifadhi taarifa muhimu.
- **Viandishi vya wakala** hutoa kumbukumbu ya nje inayodumu ambayo haiathiriwi na upunguzaji wowote wa mazungumzo.

### Uliyounda
- **Wakala mwenye ufahamu wa muktadha** anayehifadhi mfuatano katika mazungumzo yenye mizunguko mingi
- **Zana ya muhtasari** (`summarize_preferences`) inayorekodi maelezo muhimu ya mtumiaji kwa muundo mfupi
- **Mazungumzo yenye mizunguko mingi** yanaonyesha uhifadhi wa muktadha na utunzaji wa mabadiliko

### Matumizi Halisi Duniani
- **Roboti za Huduma kwa Wateja**: Kumbuka upendeleo katika vikao virefu vya msaada
- **Msaidizi wa Kibinafsi**: Fuata miradi inayoendelea bila kufafanua tena muktadha
- **Walimu wa Elimu**: Hifadhi maendeleo ya mwanafunzi katika mwingiliano mingi

### Hatua Zifuatazo
- Tekeleza zana kamili ya viandishi yenye uhifadhi wa faili
- Ongeza upunguzaji wa historia moja kwa moja baada ya muhtasari
- Changanya na hifadhidata za vector kwa ajili ya utafutaji wa kumbukumbu za maana
- Tengeneza mawakala wanaoweza kuendelea na mazungumzo siku baada ya siku kwa muktadha kamili


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
